Importing modules

In [75]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import seaborn as sns
from pathlib import Path

Defining parameters

In [76]:


n_users = 35        # secondary users
n_primary = 8       # primary channels
n_channels = 40     # available spectrum bands

noise = 0.1         # noise for SINR

PS_penalty = 75     # penalty for primary-econdary user channel collision
SS_penalty = 25     # penalty for secondary-secondary user channel collision

N=200               # particles
S=n_channels-1      # search space [0, n_channels-1]
D=n_users           # dimensions = number of users
n_iter = 600        # number of updates of positions

a=0.9
b=1.496
b_hat=1.496
c=0.5

beta_start = 1
beta_end = 0.5


Getting SINR from random

In [77]:
# Simulate channel gains randomly (in reality this comes from pathloss model)
# SINR[i][j] = signal quality if user i uses channel j
np.random.seed(67)
channel_gain = np.random.uniform(0.1, 1.0, (n_users, n_channels))     # to get SINR which gets throughput
SINR = channel_gain / noise                                           # simplified, no inter-user interference yet



Setting up Primary Users

In [78]:
# Primary user occupancy: PU[j] = 1 means channel j is occupied by primary user

PU_occupied = np.zeros(n_channels, dtype=int)
PU_occupied[np.random.choice(n_channels, n_primary, replace=False)] = 1

Fitness function which gets the throughput using SINR and adds penalty

In [79]:
def fitness(x, PS_penalty=PS_penalty, SS_penalty=SS_penalty):
    channels = np.clip(np.round(x).astype(int), 0, n_channels - 1)
    pu_mask = PU_occupied[channels]                          # 1 where PU is present
    sinr_vals = SINR[np.arange(n_users), channels]          # SINR for each user's channel
    
    throughput = np.sum(np.log2(1 + sinr_vals) * (1 - pu_mask))
    ps_penalty = np.sum(pu_mask) * PS_penalty
    
    counts = np.bincount(channels, minlength=n_channels)
    ss_pen = np.sum(np.maximum(counts - 1, 0)) * SS_penalty
    
    return -throughput + ps_penalty + ss_pen

PSO algorithm

In [80]:
def dpso(f, D, N, S, n_iter, a, b, b_hat, c, see):
    np.random.seed(see)
    x = np.random.uniform(0, S, size=(N, D))                 # setting up the initial positions of the N number of particles
    v = np.random.normal(size=(N, D))                        # setting up the initial velocities
    p = x.copy()                                             # best particle position
    fp = np.array([f(p[i]) for i in range(N)])               # throughput of all particles
    p_hat = x[np.argmin(fp)].copy()                          # global best position
    fp_hat = f(p_hat)                                        # throughput of global best position
    fp_hat_his = []

    for i in range (n_iter):
        fp_hat_his.append(float(fp_hat))
        #if i%(n_iter//10)== 0:                               # to show progress
            #print(f"progress {(i/n_iter)*100:.0f}%")
            #pass
            

        r,r_hat = np.random.uniform(0, 1, (2,N, D))          # setting up random parameters

        v = a*v + b*r*(p-x) + b_hat*r_hat*(p_hat-x)          # updating velocities as vector sum of the directions of initial velocity, local minima, local maxima
        x = x + c*v                                          # updating position according to velocities
        x = np.clip(x, 0, S)                                 # to limit the range within the subspace

        for n in range(N):                                   # calculation for each particle

            xn = x[n]                                        # getting position of that particle           
            fxn = f(xn)                                      # current throughput of that particle
            fpn = fp[n]                                      # best throughput of that particle

            if fxn < fpn:                                    # if current throughput of that particle is better the previous ones, update
                p[n] = xn.copy()
                fp[n] = fxn

                if fxn < fp_hat:                             # if the current througput is the global best throughput, update
                    p_hat = xn.copy()
                    fp_hat = fxn
    
    # print("progress 100%")
    return p_hat,fp_hat_his                                  # "coordinates", ie channel allocation of global best throughput

        



Calling PSO 

In [81]:
'''
n_seeds = 75
n_parameter = 400

results = np.empty((n_seeds, n_parameter//10))
for s in range(n_seeds):
    print("seed", s)
    for ss_idx, SS in enumerate(range(0,n_parameter,10)):
        result, history = dpso(fitness, D, SS+1, S, n_iter, a, b, b_hat, c,s)
        C_best_assignment = np.clip(np.round(result).astype(int), 0, n_channels-1)
        C_throughput = 0
        for user in range(n_users):
            ch = C_best_assignment[user]
            if not PU_occupied[ch]:
                C_throughput += np.log2(1 + SINR[user, ch])
        results[s, ss_idx] = C_throughput   
avg_throughput = np.mean(results, axis=0)



'''

'\nn_seeds = 75\nn_parameter = 400\n\nresults = np.empty((n_seeds, n_parameter//10))\nfor s in range(n_seeds):\n    print("seed", s)\n    for ss_idx, SS in enumerate(range(0,n_parameter,10)):\n        result, history = dpso(fitness, D, SS+1, S, n_iter, a, b, b_hat, c,s)\n        C_best_assignment = np.clip(np.round(result).astype(int), 0, n_channels-1)\n        C_throughput = 0\n        for user in range(n_users):\n            ch = C_best_assignment[user]\n            if not PU_occupied[ch]:\n                C_throughput += np.log2(1 + SINR[user, ch])\n        results[s, ss_idx] = C_throughput   \navg_throughput = np.mean(results, axis=0)\n\n\n\n'

In [82]:
'''

sns.set_theme(style="darkgrid")

# actual n_iter values used: SS+1 for SS in range(0, n_parameter, 10)
iter_values = list(range(1, 401, 10))   # SS+1 for SS in range(0, 800, 10)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(iter_values, avg_throughput, color="#2196F3", linewidth=2)

ax.set_xlabel("Number of Particles", fontsize=11)
ax.set_ylabel("Avg Throughput", fontsize=11)
ax.set_title("PSO — Number of Particles vs Throughput\n(averaged over 75 seeds)",fontsize=12, fontweight="bold")

ymin = np.floor(min(avg_throughput))
ymax = np.ceil(max(avg_throughput))
ax.set_yticks(np.arange(ymin, ymax + 1, 2))


plt.tight_layout()
plt.savefig(
    Path.home() / "OneDrive" / "Desktop" / "pso_N_sweep.png",
    dpi=400,
    bbox_inches="tight"
)
plt.show()
'''

'\n\nsns.set_theme(style="darkgrid")\n\n# actual n_iter values used: SS+1 for SS in range(0, n_parameter, 10)\niter_values = list(range(1, 401, 10))   # SS+1 for SS in range(0, 800, 10)\n\nfig, ax = plt.subplots(figsize=(8, 5))\nax.plot(iter_values, avg_throughput, color="#2196F3", linewidth=2)\n\nax.set_xlabel("Number of Particles", fontsize=11)\nax.set_ylabel("Avg Throughput", fontsize=11)\nax.set_title("PSO — Number of Particles vs Throughput\n(averaged over 75 seeds)",fontsize=12, fontweight="bold")\n\nymin = np.floor(min(avg_throughput))\nymax = np.ceil(max(avg_throughput))\nax.set_yticks(np.arange(ymin, ymax + 1, 2))\n\n\nplt.tight_layout()\nplt.savefig(\n    Path.home() / "OneDrive" / "Desktop" / "pso_N_sweep.png",\n    dpi=400,\n    bbox_inches="tight"\n)\nplt.show()\n'

In [83]:
'''
np.save(Path.home() / "OneDrive" / "Desktop" / "Number_of_Particles.npy", iter_values)
np.save(Path.home() / "OneDrive" / "Desktop" / "avg_throughput_N.npy", avg_throughput)
'''

'\nnp.save(Path.home() / "OneDrive" / "Desktop" / "Number_of_Particles.npy", iter_values)\nnp.save(Path.home() / "OneDrive" / "Desktop" / "avg_throughput_N.npy", avg_throughput)\n'

QPSO algorithm

In [84]:
def dqpso(f, D, N, S, n_iter, beta_start, beta_end,see):
    np.random.seed(see)
    x = np.random.uniform(0, S, size=(N, D))                            # setting up the initial positions of the N number of particles
    p = x.copy()                                                        # best particle position
    fp = np.array([f(p[i]) for i in range(N)])                          # throughput of all particles
    p_hat = x[np.argmin(fp)].copy()                                     # global best position
    fp_hat = f(p_hat)                                                   # throughput of global best position
    fp_hat_his = []

    for i in range(n_iter):
        fp_hat_his.append(float(fp_hat))
        #if i%(n_iter//10)== 0:                                          # to show progress
            #print(f"progress {(i/n_iter)*100:.0f}%")
        
        
        beta = beta_start - (beta_start - beta_end) * i / n_iter        # Beta decreases linearly from beta_start to beta_end

        mbest = np.mean(p, axis=0)                                      # Mean best position ie average of all personal bests

        phi = np.random.uniform(0,1, (N,D))                             
        p_local = phi * p + (1 - phi) * p_hat                           # local attractor for each particle (works like velocity or inertia)

        u = np.random.uniform(1e-12, 1, (N,D))                           
        sign = 2 * np.random.randint(0, 2, size=(N,D)) - 1
        x = p_local + sign * beta * np.abs(mbest - x) * np.log(1/u)     # calculates the next value of x
        x = np.clip(x, 0, S)                                            # to limit the range within the subspace


        for n in range(N):                                              # calculation for each particle

            xn = x[n]                                                   # getting position of that particle           
            fxn = f(xn)                                                 # current throughput of that particle
            fpn = fp[n]                                                 # best throughput of that particle

            if fxn < fpn:                                               # if current throughput of that particle is better the previous ones, update
                p[n] = xn.copy()
                fp[n] = fxn

                if fxn < fp_hat:                                        # if the current througput is the global best throughput, update
                    p_hat = xn.copy()
                    fp_hat = fxn
    
    #print("progress 100%")
    return p_hat, fp_hat_his                                                       # "coordinates", ie channel allocation of global best throughput

            

    

Calling QPSO

In [85]:

cases = [
    (0.25, 0.70),
    (0.15, 0.70),
    (0.25, 0.65),
    (0.15, 0.60),
    (0.25, 0.60),
    (0.25, 0.55),
    (0.15, 0.65),
    (0.35, 0.60),
    (0.20, 0.70),
    (0.10, 0.70),
]

n_seeds = 100

# shape: (seed, case)
results = np.full((n_seeds, len(cases)), np.nan)

for s in range(n_seeds):
    print(f"seed {s}")

    for ci, (beta_start, beta_end) in enumerate(cases):

        result, _ = dqpso(
            fitness,
            D,
            N,
            S,
            n_iter,
            beta_start,
            beta_end,
            s
        )

        best = np.clip(np.round(result).astype(int), 0, n_channels - 1)

        throughput = sum(
            np.log2(1 + SINR[user, best[user]])
            for user in range(n_users)
            if not PU_occupied[best[user]]
        )

        results[s, ci] = throughput

np.save(
    Path.home() / "OneDrive" / "Desktop" / "qpso_top10_results.npy",
    results
)

# ----- Analysis -----

mean = np.mean(results, axis=0)
std = np.std(results, axis=0)

ranking = np.argsort(mean)[::-1]

print(f"\n{'Rank':<5}{'beta_start':<12}{'beta_end':<10}{'Mean':<10}{'Std'}")
print("-"*55)

for rank, idx in enumerate(ranking, start=1):
    bs, be = cases[idx]
    print(f"{rank:<5}{bs:<12}{be:<10}{mean[idx]:<10.3f}{std[idx]:.3f}")

seed 0
seed 1
seed 2
seed 3
seed 4
seed 5
seed 6
seed 7
seed 8
seed 9
seed 10
seed 11
seed 12
seed 13
seed 14
seed 15
seed 16
seed 17
seed 18
seed 19
seed 20
seed 21
seed 22
seed 23
seed 24
seed 25
seed 26
seed 27
seed 28
seed 29
seed 30
seed 31
seed 32
seed 33
seed 34
seed 35
seed 36
seed 37
seed 38
seed 39
seed 40
seed 41
seed 42
seed 43
seed 44
seed 45
seed 46
seed 47
seed 48
seed 49
seed 50
seed 51
seed 52
seed 53
seed 54
seed 55
seed 56
seed 57
seed 58
seed 59
seed 60
seed 61
seed 62
seed 63
seed 64
seed 65
seed 66
seed 67
seed 68
seed 69
seed 70
seed 71
seed 72
seed 73
seed 74
seed 75
seed 76
seed 77
seed 78
seed 79
seed 80
seed 81
seed 82
seed 83
seed 84
seed 85
seed 86
seed 87
seed 88
seed 89
seed 90
seed 91
seed 92
seed 93
seed 94
seed 95
seed 96
seed 97
seed 98
seed 99

Rank beta_start  beta_end  Mean      Std
-------------------------------------------------------
1    0.25        0.65      106.424   2.845
2    0.25        0.7       106.254   3.008
3    0.15        0.65     

In [86]:
'''
beta_start_values = [0.10,0.15,0.20,0.25,0.30,0.35,0.40]
beta_end_values   = [0.40,0.45,0.50,0.55,0.60,0.65,0.70]
n_seeds = 25

results = np.full((n_seeds, len(beta_start_values), len(beta_end_values)), np.nan)

for s in range(n_seeds):
    print(f"seed {s}")
    for bsi, beta_start in enumerate(beta_start_values):
        for bei, beta_end in enumerate(beta_end_values):
            result, _ = dqpso(fitness, D, N, S, n_iter, beta_start, beta_end, s)
            best = np.clip(np.round(result).astype(int), 0, n_channels - 1)
            throughput = sum(
                np.log2(1 + SINR[user, best[user]])
                for user in range(n_users)
                if not PU_occupied[best[user]]
            )
            results[s, bsi, bei] = throughput

np.save(Path.home() / "OneDrive" / "Desktop" / "qpso_grid_results3.npy", results)

avg = np.mean(results, axis=0)   # shape: (beta_start, beta_end)

best_idx = np.unravel_index(np.argmax(avg), avg.shape)
print(f"\nBest combination:")
print(f"  beta_start = {beta_start_values[best_idx[0]]}")
print(f"  beta_end   = {beta_end_values[best_idx[1]]}")
print(f"  avg throughput = {avg[best_idx]:.3f}")
'''


'\nbeta_start_values = [0.10,0.15,0.20,0.25,0.30,0.35,0.40]\nbeta_end_values   = [0.40,0.45,0.50,0.55,0.60,0.65,0.70]\nn_seeds = 25\n\nresults = np.full((n_seeds, len(beta_start_values), len(beta_end_values)), np.nan)\n\nfor s in range(n_seeds):\n    print(f"seed {s}")\n    for bsi, beta_start in enumerate(beta_start_values):\n        for bei, beta_end in enumerate(beta_end_values):\n            result, _ = dqpso(fitness, D, N, S, n_iter, beta_start, beta_end, s)\n            best = np.clip(np.round(result).astype(int), 0, n_channels - 1)\n            throughput = sum(\n                np.log2(1 + SINR[user, best[user]])\n                for user in range(n_users)\n                if not PU_occupied[best[user]]\n            )\n            results[s, bsi, bei] = throughput\n\nnp.save(Path.home() / "OneDrive" / "Desktop" / "qpso_grid_results3.npy", results)\n\navg = np.mean(results, axis=0)   # shape: (beta_start, beta_end)\n\nbest_idx = np.unravel_index(np.argmax(avg), avg.shape)\nprint

In [87]:
'''
# --- single heatmap ---
sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    avg,
    ax=ax,
    xticklabels=[f"{v}" for v in beta_end_values],
    yticklabels=[f"{v}" for v in beta_start_values],
    annot=True, fmt=".1f",
    cmap="YlOrRd",
    vmin=avg.min(), vmax=avg.max()
)
ax.set_xlabel("beta_end", fontsize=11)
ax.set_ylabel("beta_start", fontsize=11)
ax.set_title(f"QPSO Grid Search — Avg Throughput ({n_seeds} seeds)",
             fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig(
    Path.home() / "OneDrive" / "Desktop" / "qpso_grid_heatmap3.png",
    dpi=400, bbox_inches="tight"
)
plt.show()
'''

'\n# --- single heatmap ---\nsns.set_theme(style="white")\n\nfig, ax = plt.subplots(figsize=(7, 5))\nsns.heatmap(\n    avg,\n    ax=ax,\n    xticklabels=[f"{v}" for v in beta_end_values],\n    yticklabels=[f"{v}" for v in beta_start_values],\n    annot=True, fmt=".1f",\n    cmap="YlOrRd",\n    vmin=avg.min(), vmax=avg.max()\n)\nax.set_xlabel("beta_end", fontsize=11)\nax.set_ylabel("beta_start", fontsize=11)\nax.set_title(f"QPSO Grid Search — Avg Throughput ({n_seeds} seeds)",\n             fontsize=12, fontweight="bold")\n\nplt.tight_layout()\nplt.savefig(\n    Path.home() / "OneDrive" / "Desktop" / "qpso_grid_heatmap3.png",\n    dpi=400, bbox_inches="tight"\n)\nplt.show()\n'

In [88]:
'''
flat = avg.ravel()

top10 = np.argsort(flat)[::-1][:10]

print(f"{'Rank':<5}{'beta_start':<12}{'beta_end':<10}{'Avg Throughput'}")
print("-"*45)

cases = []

for rank, idx in enumerate(top10, start=1):

    si, ei = np.unravel_index(idx, avg.shape)

    beta_start = beta_start_values[si]
    beta_end   = beta_end_values[ei]
    throughput = avg[si, ei]

    cases.append((beta_start, beta_end))

    print(f"{rank:<5}{beta_start:<12}{beta_end:<10}{throughput:.3f}")
    '''

'\nflat = avg.ravel()\n\ntop10 = np.argsort(flat)[::-1][:10]\n\nprint(f"{\'Rank\':<5}{\'beta_start\':<12}{\'beta_end\':<10}{\'Avg Throughput\'}")\nprint("-"*45)\n\ncases = []\n\nfor rank, idx in enumerate(top10, start=1):\n\n    si, ei = np.unravel_index(idx, avg.shape)\n\n    beta_start = beta_start_values[si]\n    beta_end   = beta_end_values[ei]\n    throughput = avg[si, ei]\n\n    cases.append((beta_start, beta_end))\n\n    print(f"{rank:<5}{beta_start:<12}{beta_end:<10}{throughput:.3f}")\n    '

In [89]:
'''
print("Classical Particle Swarm Optimization\n")
print("Channel assignment:", C_best_assignment)
print("Throughput:", C_throughput)

print("\n\nQuantum Particle Swarm Optimization \n")
print("Channel assignment:", Q_best_assignment)
print("Throughput:", Q_throughput)
'''

'\nprint("Classical Particle Swarm Optimization\n")\nprint("Channel assignment:", C_best_assignment)\nprint("Throughput:", C_throughput)\n\nprint("\n\nQuantum Particle Swarm Optimization \n")\nprint("Channel assignment:", Q_best_assignment)\nprint("Throughput:", Q_throughput)\n'